# 实战案例：基于对应表示的跨模态检索

## 文件目录与提交规范

**本作业要求使用PyTorch实现。请补全标有 `# TODO` 的代码。**

所有路径均相对于本作业文件夹。原始数据固定放在 `data/flickr8k/`，其中包含 `dataset_flickr8k.json` 和 `images/`。生成的 `vocab.json`、`train_data.json`、`val_data.json`、`test_data.json` 也保存在 `data/flickr8k/`。

模型统一保存在 `work/` 文件夹中，模型文件名自行确定；代码中的文件名仅为示例，可自行修改。运行时自动创建 `work/`。提交完成的 Notebook 与 `work/` 文件夹，不提交 `data/` 数据集。

## 跨模态检索技术简介

![计算图文匹配分数的两类方法](img/cr-corr-vs-joint.png)

跨模态检索的关键就是建立不同模态数据之间的关联，更直接地，模型需要能够输出多个模态数据的匹配分数。如图所示，现有的方法可以被分为两类：一是学习图文多模态对应表示，然后直接利用图像和文本的对应表示的距离计算匹配分数，我们称这类模型为对应表示方法；二是学习图文多模态共享表示，然后在共享表示层上增加一个或多个网络层直接输出图像和文本的匹配分数，我们称这类模型为共享表示方法。

一般而言，和对应表示方法相比，共享表示方法因为充分融合了图文信息，可以获得更好的性能。一个直观的理解是给定两个模态的数据，对应表示方法限定了两个模态的关联必须是在没有交互的前提下建立，而共享表示方法则没有该限制。因此，共享表示方法拥有更大的自由度来拟合数据的分布。

然而，共享表示方法的检索非常耗时。例如，在执行以文检图任务中，需要将文本查询和候选集中的每一张图片都成对的输入到模型中，才能得到文本查询与候选集中所有图片的匹配分数。
而对应表示方法只需要提前离线计算好候选集中所有图片的表示，然后在检索时，只需要实时计算文本查询的表示，再利用最近邻检索算法搜索图像最近邻即可。
因此，对应表示方法在实际的跨模态检索中使用更为广泛。

接下来，我们将具体介绍使用对应表示方法的模型VSE++的实战案例，其官方代码见[链接](https://github.com/fartashf/vsepp)。为了使读者更清晰的理解模型的训练过程，我们重新实现了该模型。

## 1. 环境初始化与数据定位

本单元导入 PyTorch、NumPy、PIL 等依赖，固定随机种子并定位作业目录。默认以当前工作目录作为根目录，也可通过代码中的作业目录环境变量指定根目录。

请提前将图片和划分文件整理到 `data/flickr8k/`：`images/` 存放图片，`dataset_flickr8k.json` 保存图片、caption 和官方 split。程序会检查这两个输入；缺失时会提示错误。本单元不执行解压操作。

使用 Karpathy 提供的标准划分：训练集6000张图片、验证集1000张、测试集1000张，每图5条描述，原始文本最多保留30个 token。运行后应先核对打印的数据来源。

In [ ]:
from pathlib import Path
import json
import math
import os
import random
from collections import Counter, defaultdict
from types import SimpleNamespace

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
configured_root = os.environ.get("HOMEWORK3_DIR")
NOTEBOOK_DIR = (
    Path(configured_root).expanduser().resolve()
    if configured_root else Path.cwd().resolve()
)
SOURCE_DIR = NOTEBOOK_DIR / "data" / "flickr8k"
if not (SOURCE_DIR / "dataset_flickr8k.json").is_file() or not (SOURCE_DIR / "images").is_dir():
    raise FileNotFoundError(
        f"请将数据放在 {SOURCE_DIR}，该目录须包含 dataset_flickr8k.json 和 images/。"
    )
OUTPUT_DIR = SOURCE_DIR
CAPTIONS_PER_IMAGE = 5
MAX_LEN = 30
print(f"使用设备: {device}")
print(f"数据来源: {SOURCE_DIR}")

## 2. 整理数据集与构建词表

原始 JSON 中的 `split` 定义了训练、验证和测试划分（6000/1000/1000）。将自然语言描述编码为整数序列；图片只记录路径，读取样本时再加载图像。

词表只统计训练集 caption，保留出现至少5次的词，并加入 `<pad>`、`<unk>`、`<start>`、`<end>`。验证和测试中的未登录词映射为 `<unk>`，不能参与词表构建。文本截断后在首尾加入开始和结束标记。

输出 `vocab.json`、`train_data.json`、`val_data.json`、`test_data.json`，均保存在 `data/flickr8k/`。生成30000条训练描述和各5000条验证、测试描述。

**实现说明：** 对照 TODO 1 完成按 split 收集图片，理解图片列表与 caption 列表的对应关系，并核对生成数量。

In [ ]:
def prepare_dataset(source_dir, output_dir, min_word_count=5, max_len=30):
    """按 Karpathy split 使用完整数据集，生成词表和编码数据。"""
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw = json.loads((source_dir / "dataset_flickr8k.json").read_text())
    by_split = defaultdict(list)
    for item in raw["images"]:
        split = item.get("split")
        if split in ("train", "val", "test"):
            by_split[split].append(item)

    # TODO 1：按 split 收集所有图片
    selected_by_split = {split: items for split, items in by_split.items()}

    vocab_counter = Counter()
    for item in selected_by_split["train"]:
        for sentence in item["sentences"][:CAPTIONS_PER_IMAGE]:
            vocab_counter.update(sentence["tokens"][:max_len])
    words = sorted(word for word, count in vocab_counter.items() if count >= min_word_count)
    vocab = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
    vocab.update({word: index + 4 for index, word in enumerate(words)})
    (output_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False))

    summary = {}
    for split, items in selected_by_split.items():
        image_paths, encoded_captions = [], []
        for item in items:
            image_path = source_dir / "images" / item["filename"]
            if not image_path.is_file():
                raise FileNotFoundError(image_path)
            image_paths.append(str(image_path))
            sentences = item["sentences"][:CAPTIONS_PER_IMAGE]
            if len(sentences) < CAPTIONS_PER_IMAGE:
                sentences = sentences + [sentences[-1]] * (CAPTIONS_PER_IMAGE - len(sentences))
            for sentence in sentences:
                tokens = sentence["tokens"][:max_len]
                encoded = [vocab["<start>"]]
                encoded += [vocab.get(token, vocab["<unk>"]) for token in tokens]
                encoded += [vocab["<end>"]]
                encoded_captions.append(encoded)
        payload = {"IMAGES": image_paths, "CAPTIONS": encoded_captions}
        (output_dir / f"{split}_data.json").write_text(json.dumps(payload))
        summary[split] = (len(image_paths), len(encoded_captions))
    return vocab, summary


vocab, split_summary = prepare_dataset(SOURCE_DIR, OUTPUT_DIR, min_word_count=5, max_len=MAX_LEN)
print("数据划分:", split_summary, "词表大小:", len(vocab))

## 3. 定义 Dataset 与批量读取

自定义数据集继承 `torch.utils.data.Dataset`，实现 `__len__` 和 `__getitem__`。`ImageTextDataset` 以 caption 为样本单位；每张图片连续对应5条 caption，因此 caption 下标整除5得到图片下标。

图像使用 ImageNet 预处理：缩放到256×256后裁剪到224×224（训练随机裁剪，验证/测试中心裁剪），转为 tensor 并按 ImageNet 均值和标准差归一化。caption 用 `<pad>` 补齐至32个位置（max_len + 2），同时返回包含开始与结束标记的真实长度。

`make_loaders` 为训练、验证和测试分别构造 DataLoader；训练打乱顺序，评估保持顺序。`UniqueImageCaptionDataset` 额外提供每图仅一条描述的采样方式。

**实现说明：** TODO 2 需要完成 caption 到图片的索引映射；不要把 caption 数误认为唯一图片数。

In [ ]:
train_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.RandomCrop(224),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
])

eval_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.CenterCrop(224),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
])


def image_to_tensor(path, transform=None):
    image = Image.open(path).convert("RGB")
    if transform is not None:
        return transform(image)
    image = image.resize((224, 224))
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


class ImageTextDataset(Dataset):
    def __init__(self, data_path, vocab_path, captions_per_image=5, max_len=30, transform=None):
        self.data = json.loads(Path(data_path).read_text())
        self.vocab = json.loads(Path(vocab_path).read_text())
        self.cpi = captions_per_image
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.data["CAPTIONS"])

    def __getitem__(self, index):
        # TODO 2：由 caption 下标映射到 image 下标
        image_index = index // self.cpi
        image = image_to_tensor(self.data["IMAGES"][image_index], self.transform)
        tokens = self.data["CAPTIONS"][index]
        length = len(tokens)
        padded = tokens + [self.vocab["<pad>"]] * (self.max_len + 2 - length)
        return image, torch.tensor(padded, dtype=torch.long), length


class UniqueImageCaptionDataset(ImageTextDataset):
    """检索训练集：每张图片只取一条 caption，避免同图 caption 成为假负例。"""
    def __len__(self):
        return len(self.data["IMAGES"])

    def __getitem__(self, image_index):
        caption_index = image_index * self.cpi + image_index % self.cpi
        image, caption, length = super().__getitem__(caption_index)
        return image, caption, length, self.data["IMAGES"][image_index]


def make_loaders(output_dir, batch_size=32, unique_train_images=False):
    output_dir = Path(output_dir)
    vocab_path = output_dir / "vocab.json"
    generator = torch.Generator().manual_seed(SEED)
    datasets = {}
    for split in ("train", "val", "test"):
        tf = train_transform if split == "train" else eval_transform
        datasets[split] = ImageTextDataset(
            output_dir / f"{split}_data.json", vocab_path,
            CAPTIONS_PER_IMAGE, MAX_LEN, transform=tf,
        )
    train_dataset = datasets["train"]
    if unique_train_images:
        train_dataset = UniqueImageCaptionDataset(
            output_dir / "train_data.json", vocab_path,
            CAPTIONS_PER_IMAGE, MAX_LEN, transform=train_transform,
        )
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                   num_workers=0, generator=generator),
        DataLoader(datasets["val"], batch_size=batch_size, shuffle=False, num_workers=0),
        DataLoader(datasets["test"], batch_size=batch_size, shuffle=False, num_workers=0),
    )

## 4. 图像编码器 ImageEncoder（预训练 CNN）

本实现使用 torchvision 提供的预训练 CNN，支持两种骨干网络：

- **ResNet-152**：去掉最后的全连接层和平均池化层，卷积输出为2048维，通过自适应平均池化获得全局特征向量。
- **VGG-19**：使用完整网络（包含 classifier 部分），去掉最后一个分类层，输出为4096维全局特征。

两种骨干网络均在 ImageNet 上预训练，能够提取高质量的视觉语义特征。增加一个线性投影层将特征维度映射到 `embed_dim` 维，使图像表示与文本表示维度一致。

`grid=True` 返回局部网格特征（供注意力模型使用）；`grid=False` 返回 `[B, embed_dim]` 的全局向量（供检索模型使用）。跨模态检索使用 `grid=False`。

`finetuned=True` 时骨干网络参数参与梯度更新（微调），`finetuned=False` 时冻结参数。学生可任选 ResNet-152 或 VGG-19 作为骨干网络。

**实现说明：** TODO 3 完成骨干网络特征提取与线性投影。注意不同骨干网络的特征维度不同（ResNet-152 为2048，VGG-19 为4096），需要正确设置投影层输入维度。

In [ ]:
class ImageEncoder(nn.Module):
    """预训练 CNN 图像编码器，支持 ResNet-152 和 VGG-19 骨干网络。"""
    def __init__(self, embed_dim=1024, grid=True, finetuned=True, backbone="resnet152"):
        super().__init__()
        self.grid = grid
        self.backbone = backbone
        if backbone == "resnet152":
            resnet = tv_models.resnet152(weights=tv_models.ResNet152_Weights.IMAGENET1K_V1)
            self.features = nn.Sequential(*(list(resnet.children())[:-2]))
            self.feat_dim = 2048
        elif backbone == "vgg19":
            vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.IMAGENET1K_V1)
            vgg.classifier = nn.Sequential(*list(vgg.classifier.children())[:-1])
            self.features = vgg
            self.feat_dim = 4096
        else:
            raise ValueError(f"不支持的骨干网络: {backbone}，请选择 'resnet152' 或 'vgg19'")
        for param in self.features.parameters():
            param.requires_grad = finetuned
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if not grid:
            self.projection = nn.Linear(self.feat_dim, embed_dim)

    def forward(self, images):
        # TODO 3：提取骨干网络特征，按 grid 参数返回网格或全局投影特征
        features = self.features(images)
        if self.grid:
            return features
        if features.dim() == 2:
            return self.projection(features)
        pooled = self.global_pool(features).flatten(1)
        return self.projection(pooled)


TinyImageEncoder = ImageEncoder

## 5. 文本表示、VSE++ 与 Hard Negative Loss

### 文本编码与双编码器

`TextRepExtractor` 将 token ID 映射为300维词向量。GRU 使用真实序列长度打包变长输入（TODO 4），其最后一层 hidden state 作为1024维文本表示。`enforce_sorted=False` 允许输入长度不预排序。

`VSEPP` 分别编码图片和文本，对两个输出做 L2 normalization，获得对应表示空间中的单位向量。因此内积等于 cosine similarity，避免向量范数主导匹配分数。

### 在线困难负样本挖掘

对于 batch 内 B 个配对样本，相似度矩阵为 $S_{ij}=v_i^Tt_j$，对角线对应匹配图文。双向 margin loss 分别沿图像和文本查询方向选取最违反间隔的负样本：

$$L=\sum_i\max_{j\ne i}[m+S_{ij}-S_{ii}]_+
+\sum_i\max_{j\ne i}[m+S_{ji}-S_{ii}]_+.$$

当前 margin 为0.2。先计算双向代价，屏蔽对角正例，再分别取行、列最大值，最后求两个方向的损失之和。TODO 5 实现这一完整过程。

`TripletNetLoss` 支持 `hard_negative` 参数：当 `hard_negative=True` 时，取每行/列最大代价（hardest-negative 策略）；当 `hard_negative=False` 时，对所有非对角元素求和（普通 triplet loss）。通过对比两种模式的效果，可以理解困难负样本挖掘对 VSE++ 性能的影响。

训练启用 `unique_train_images=True`，每张图片选择一条固定 caption，共6000个训练 pair，避免同图其他合法 caption 被视为负样本。验证与测试仍保留每图全部5条描述。

In [ ]:
class TextRepExtractor(nn.Module):
    def __init__(self, vocab_size, word_dim=300, embed_dim=1024):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.gru = nn.GRU(word_dim, embed_dim, batch_first=True)

    def forward(self, captions, lengths):
        embedded = self.embedding(captions)
        # TODO 4：按真实长度打包序列
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        return F.normalize(hidden[-1], dim=1)


class VSEPP(nn.Module):
    def __init__(self, vocab_size, embed_dim=1024, word_dim=300, backbone="resnet152"):
        super().__init__()
        self.image_extractor = ImageEncoder(embed_dim, grid=False, finetuned=True, backbone=backbone)
        self.text_extractor = TextRepExtractor(vocab_size, word_dim, embed_dim)

    def forward(self, images, captions, lengths):
        image_code = F.normalize(self.image_extractor(images), dim=1)
        text_code = self.text_extractor(captions, lengths)
        return image_code, text_code


class TripletNetLoss(nn.Module):
    def __init__(self, margin=0.2, hard_negative=False):
        super().__init__()
        self.margin = margin
        self.hard_negative = hard_negative

    def forward(self, image_code, text_code):
        # TODO 5：实现双向 margin ranking 与 hard negative 挖掘
        scores = image_code @ text_code.t()
        diagonal = scores.diag()
        image_cost = (self.margin + scores - diagonal.unsqueeze(1)).clamp_min(0)
        text_cost = (self.margin + scores - diagonal.unsqueeze(0)).clamp_min(0)
        eye = torch.eye(scores.size(0), dtype=torch.bool, device=scores.device)
        image_cost = image_cost.masked_fill(eye, 0)
        text_cost = text_cost.masked_fill(eye, 0)
        if self.hard_negative:
            image_cost = image_cost.max(dim=1).values
            text_cost = text_cost.max(dim=0).values
        return image_cost.sum() + text_cost.sum()

## 6. 训练参数、模型选择与 Recall@K

### 参数在哪里设置

下方代码单元的配置包括 batch size、训练轮数、学习率、margin、骨干网络和是否使用 hard negative 等。训练使用 Adam 优化器，学习率采用分段衰减方法（每隔 `lr_update` 轮衰减为原来的一半），梯度裁剪阈值为2。可将 `backbone` 改为 `"vgg19"` 切换至 VGG-19 骨干网络。

训练流程为读取唯一图文 pair → 编码图文（TODO 6）→ 双向 triplet loss → 反向传播 → 梯度裁剪 → Adam 更新。每轮评估验证集，根据 Rsum 选择最佳参数，最后加载最佳参数进行测试，并在 `work/` 保存模型。模型文件名自行确定。

### 默认参数与调参要求

**上述参数仅为默认配置，并非最优参数。完成本作业时，需要调整训练参数，通过实验争取更好的模型效果，不能只使用默认参数运行一次后直接提交。**

可调整训练轮数（`epochs`）、学习率（`learning_rate`）、批大小（`batch_size`）、骨干网络（`backbone`）、间隔参数（`margin`）和是否使用困难负样本挖掘（`hard_negative`）等。增加训练轮数并不一定提高效果，应结合训练损失和验证集指标判断是否过拟合。

请保留默认配置作为基线，至少尝试一组调整后的配置，并记录各组参数、训练过程和验证集指标。依据验证集结果选择最终配置和模型，再在测试集上进行最终评估；不得依据测试集结果反复调参。报告中需说明调整依据、与默认配置相比的变化，以及最终模型效果；若未获得提升，应如实报告并分析原因。

### Recall@K

按固定顺序收集图文表示后，每隔5条取一个图像向量，与全部文本向量计算相似度矩阵（TODO 7）。I2T 以图片查询文本，5条正确 caption 中任一进入前 K 即命中；T2I 以文本查询图片，检查其唯一对应图片的名次。

输出依次为 **I2T R@1、R@5、R@10，T2I R@1、R@5、R@10**，数值以百分数表示，六项之和为 Rsum。由于 Top-1 包含于 Top-5、Top-10，Recall 随 K 不应下降。

提交时保留两轮训练日志、六项测试 Recall 和 Rsum。不同方向的查询数与正确匹配数量不同，其指标可能不同。

In [ ]:
def train_retrieval_epoch(loader, model, loss_fn, optimizer):
    model.train()
    total = 0.0
    for images, captions, lengths, _image_paths in loader:
        images, captions, lengths = images.to(device), captions.to(device), lengths.to(device)
        optimizer.zero_grad()
        # TODO 6：完成图像和文本编码
        image_code, text_code = model(images, captions, lengths)
        loss = loss_fn(image_code, text_code)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
        total += loss.item() * images.size(0)
    return total / len(loader.dataset)


def calc_recall(image_codes, text_codes, captions_per_image=5):
    unique_images = image_codes[::captions_per_image]
    # TODO 7：计算所有图片与文本的检索分数
    scores = unique_images @ text_codes.T
    image_ranks = []
    for i in range(len(unique_images)):
        order = np.argsort(-scores[i])
        positives = set(range(i*captions_per_image, (i+1)*captions_per_image))
        image_ranks.append(min(rank for rank, index in enumerate(order) if index in positives))
    text_ranks = []
    image_order = np.argsort(-scores, axis=0)
    for i in range(len(text_codes)):
        text_ranks.append(int(np.where(image_order[:, i] == i//captions_per_image)[0][0]))
    recalls = []
    for ranks in (np.asarray(image_ranks), np.asarray(text_ranks)):
        recalls.extend([100.0*np.mean(ranks < k) for k in (1, 5, 10)])
    return tuple(recalls)


def evaluate_retrieval(loader, model):
    image_codes, text_codes = [], []
    model.eval()
    with torch.no_grad():
        for images, captions, lengths in loader:
            image_code, text_code = model(images.to(device), captions.to(device), lengths.to(device))
            image_codes.append(image_code.cpu().numpy())
            text_codes.append(text_code.cpu().numpy())
    return calc_recall(np.concatenate(image_codes), np.concatenate(text_codes), CAPTIONS_PER_IMAGE)


config = SimpleNamespace(
    batch_size=32, epochs=45, learning_rate=0.00002, lr_update=15,
    margin=0.2, hard_negative=True, grad_clip=2, backbone="resnet152",
)
train_loader, val_loader, test_loader = make_loaders(
    OUTPUT_DIR, config.batch_size, unique_train_images=True
)
model = VSEPP(len(vocab), backbone=config.backbone).to(device)
loss_fn = TripletNetLoss(config.margin, config.hard_negative)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=config.lr_update, gamma=0.5)
best_state, best_rsum = None, -1.0
for epoch in range(1, config.epochs + 1):
    loss = train_retrieval_epoch(train_loader, model, loss_fn, optimizer)
    scheduler.step()
    recalls = evaluate_retrieval(val_loader, model)
    rsum = sum(recalls)
    print(f"epoch {epoch}/{config.epochs}: loss={loss:.4f}, lr={optimizer.param_groups[0]['lr']:.6f}, val_Rsum={rsum:.1f}")
    if rsum >= best_rsum:
        best_rsum = rsum
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
test_recalls = evaluate_retrieval(test_loader, model)
test_rsum = sum(test_recalls)
checkpoint_dir = NOTEBOOK_DIR / "work"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
torch.save({"model": best_state, "test_recalls": test_recalls}, checkpoint_dir / f"vsepp_{config.backbone}.pt")
print(f"Backbone: {config.backbone}, Hard Negative: {config.hard_negative}")
print("Test recalls (I2T R@1/5/10, T2I R@1/5/10):", tuple(round(x, 2) for x in test_recalls))
print(f"Test Rsum={test_rsum:.1f}")

## 实验分析题参考要点

数值与失败案例须以实际运行结果为准。

### 1. 表示归一化（5分）

结合 VSEPP.forward 与相似度计算解释两种向量为什么都需要 L2 normalization。

**回答：**

归一化后内积等于 cosine similarity，匹配依据方向而不由向量范数主导。如果不做归一化，范数较大的向量在内积计算中天然占优势，模型可能通过增大范数而非学习语义对齐来降低 loss。归一化将所有表示约束到单位超球面上，使相似度仅反映语义方向的接近程度，训练更加稳定。

### 2. Hard Negative 与配对（5分）

说明双向 margin、对角线屏蔽和行列最大值的作用，并解释每图只取一条训练 caption 的原因。

**回答：**

对角线是正例对 $(v_i, t_i)$，双向代价分别从图像查询（固定图像找最难文本负例）和文本查询（固定文本找最难图像负例）两个方向衡量违反间隔的程度。屏蔽对角线确保正例不参与负例选择。当 `hard_negative=True` 时，取行/列最大值实现 hardest-negative 策略，聚焦于最具迷惑性的负样本，梯度信号更强；当 `hard_negative=False` 时，对所有非对角元素求和，相当于普通 triplet loss。对比两种模式可以验证困难负样本挖掘对检索性能的提升效果。同一张图的其他 caption 也是合法的正例匹配，如果出现在 batch 中被当作负样本会产生错误的梯度方向，因此每张图片只取一条 caption 参与训练。

### 3. Recall@K（5分）

填写 I2T 和 T2I 的六项 Recall 与 Rsum，说明为什么 R@10 不低于 R@1、两个方向结果为什么不同。

**回答：**

指标从实际测试输出填写。R@K 是 Top-K 命中率，Top-10 的候选集合包含 Top-1，因此 R@10 >= R@5 >= R@1。I2T 和 T2I 方向不同的原因：I2T 每张查询图片有5个正确 caption，只要有一个进入 Top-K 即算命中，成功概率较高；T2I 每条查询文本只有一张正确图片，且候选图片数量（1000张）远少于候选文本数量（5000条），查询数量和正确匹配条件都不同，因此两个方向的 Recall 通常不同。

### 4. 检索失败案例（5分）

可新增展示单元，给出一个失败 query、Top-5 与正确匹配位置，并结合视觉语义、文本细节或数据歧义分析原因。

**回答：**

应根据实际 query 和排序结果分析，不能用虚构结果。常见失败原因包括：（1）视觉相似的图像导致混淆，如多张包含狗的图片难以区分；（2）文本描述中的细节信息（颜色、数量、动作）难以被简单的 GRU 编码区分；（3）数据歧义——同一描述可能合理匹配多张图片。使用预训练 ResNet-152/VGG-19 后视觉表示质量显著提升，但文本编码器的语义理解能力仍然有限。